# Langchain Agentic Programming

## Imports and Constants

In [1]:
import ast, re, math, os, sys

In [2]:
from euriai.langchain import create_chat_model
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain.output_parsers.structured import StructuredOutputParser, ResponseSchema
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableWithMessageHistory, RunnableConfig
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory as ChatMessageHistory
from langchain_core.tools import BaseTool, tool
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    CombinedMemory,
    ReadOnlySharedMemory,
    ConversationEntityMemory
)

/home/rishicarter/miniforge3/envs/streamlitenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
os.environ['euri-api-key'] = "euri-c05d288a7b37c96408e0a1fac6dba00717a742ed24f3e135ffa05af0d98dc3c4"

In [4]:
llm_oss120b = create_chat_model(
    api_key=os.getenv("euri-api-key"),
    model="openai/gpt-oss-120b",
    temperature=0.2
)

In [5]:
# gpt-5-nano-2025-08-07
llm_gpt = create_chat_model(
    api_key=os.getenv("euri-api-key"),
    model="gpt-4.1-nano",
    temperature=0.2
)

## Experiment

In [ ]:
from euriai.langchain import create_chat_model

chat_model = create_chat_model(
    api_key=os.getenv("euri-api-key"),
    model="gpt-4.1-nano",
    temperature=0.7
)

response = chat_model.invoke("What is artificial intelligence?")
print(response.content)

In [ ]:
import pydantic
print(pydantic.__version__)

In [ ]:
import euriai
print(euriai.__version__)

In [ ]:
from euriai.langchain import create_chat_model

# Full streaming support
chat_model = create_chat_model(api_key=os.getenv("euri-api-key"))

# Streaming response
for chunk in chat_model.stream("Tell me a story"):
    print(chunk.content, end="", flush=True)

# Async streaming
import asyncio

async def async_stream():
    async for chunk in chat_model.astream("Tell me a story"):
        print(chunk.content, end="", flush=True)

In [ ]:
response = llm_oss120b.invoke("What is artificial intelligence?")
print(response.content)

## Agents

In [29]:
@tool("calculator", return_direct=True)
def calculator(expression: str) -> str:
    """
    Evaluate numeric math expression. Supports +, -, *, /, **, parenthesis and all kinds of mathematical functions
    """
    allowed_nodes = (
        ast.Expression,
        ast.Call,
        ast.Name,
        ast.arguments,
        ast.arg,
        ast.Pow,
        ast.UnaryOp,
        ast.unaryop,
        ast.Constant,
        ast.Load,
        ast.FunctionDef,
        ast.Module,
        ast.Expr,
        ast.BinOp,
        ast.operator,
        ast.Load,
        ast.Num
    )

    allowed_names = {k:v for k,v in vars(math).items() if not k.startswith("_")}
    allowed_names.update({"abs": abs, "round": round, "min": min, "max": max})
    node = ast.parse(expression, mode="eval")

    for n in ast.walk(node):
        if not isinstance(n, allowed_nodes):
            raise ValueError(f"Expression contains invalid node! {type(n)}")
        if not isinstance(n, ast.Name) and n.id not in allowed_names:
            raise ValueError(f"Expression contains invalid name! {type(n)}")
        
    code = compile(node, "<string>", "eval")
    return (str(eval(code, {"__builtins__": {}}, allowed_names)))

In [30]:
tool_math = [calculator]

In [31]:
math_agent_prompt = ChatPromptTemplate.from_messages([(
    "system", "You are a computing math assistant. You can use this tool:\n{tools}\n"
    "When using tool follow these instructions exactly the same format:\n"
    "Questions:...........\nthought.......\nAction:....... by using one of the tools"
    "access that you have [{tool_names}]\n"
    "Always finish with final answer which needs to a numeric answer if the question suggests artithematic operation."),
    ("human", "questions: {input}\n{agent_scratchpad}")
])

Agent Scratchpad - Reasoning by keeping context in memory

In [32]:
math_agent = create_react_agent(
    llm=llm_gpt,
    prompt=math_agent_prompt,
    tools=tool_math
)

In [33]:
math_agent_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    input_key="input",
    output_key="output"
)

In [34]:
math_ai = AgentExecutor(
    agent=math_agent,
    tools=tool_math,
    memory=math_agent_memory,
    verbose=True,
    handle_parsing_errors=(
        "Follow the instructions given to use the tools"
    )
)

* handler = workflow.run(user_msg="Can you add 5 and 3 then multiply result with 76 then add 23 then divide with 2 then add 17 then subtract 12 then multiply with 16 then divide with 5?", ctx=ctx)
* handler = workflow.run(user_msg="Can you add 5 and 3?", ctx=ctx)
* handler = workflow.run(user_msg="Can you add 5 and 3. Also tell me how I can guide to perform multiple arithematic operations using the tools because I want to ask you to perform this - Can you add 5 and 3 then multiply result with 76 then add 23 then divide with 2 then add 17 then subtract 12 then multiply with 16 then divide with 5?", ctx=ctx)
* handler = workflow.run(user_msg="Can you add 5 and 3. Now multiply result with 76. Then add 23 to result. Then divide result with 2. Then add 17 to result. Then subtract 12 from result. Then multiply result with 16. Then divide result with 5", ctx=ctx)

In [35]:
@tool("kb_search", return_direct=True)
def kb_search(query: str) -> str:
    """a mock function to search from a knowledge base"""
    knowledge_base = {
        "what is the capital of france": "The capital of France is Paris.",
        "who is the president of the united states": "The president of the United States is Joe Biden.",
        "what is the largest mammal": "The largest mammal is the blue whale.",
    }
    return knowledge_base.get(query.lower(), "I don't know the answer to that question.")

In [36]:
tool_kb = [kb_search]

In [37]:
kb_agent_prompt = ChatPromptTemplate.from_messages([
    ("system","you are a helpful assistant that can answer question based on your knowledge base and you can use the following tools:\n{tools}\n"
    "when you are going to use this tools follw these instruction exactly the same format:\n"
    "Questions :......\nthought ...\nAction by using one of the tools access that you have [{tool_names}]\n "
    "alwasy finish with final give me an answer to the question"),
    ("human","questions: {input}\n{agent_scratchpad}")
])

In [38]:
kb_agent = create_react_agent(
    llm=llm_gpt,
    prompt=kb_agent_prompt,
    tools=tool_kb
)

In [39]:
kb_agent_memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True,
    input_key="input",
    output_key="output"
)

In [40]:
kb_ai = AgentExecutor(
    agent=kb_agent,
    tools=tool_kb,
    memory=kb_agent_memory,
    verbose=True,
    handle_parsing_errors=(
        "Follow the instructions given to use the tools"
    )
)

In [41]:
router_agent_prompt = ChatPromptTemplate.from_messages([
    (
        "system", "You are a router. Read the user message and using exactly one token evaluate and output the input given by users:\n"
        "- MATH if user is asking for Calculations\n"
        "- KB if user is asking for General Knowledge\n"
        "Output only KB or MATH and nothing else."
    ),
    ("human", "{input}")
])

In [42]:
router_chain = router_agent_prompt | llm_gpt | StrOutputParser()

In [43]:
def _dispatcher(user_input: dict):
    """
    Function to take inputs from User and attach history to memory
    """
    user_msg = user_input["input"]
    choice = router_chain.invoke({"input": user_msg}).strip().upper()
    if choice == "MATH":
        return math_ai.invoke({"input": user_msg})
    elif choice == "KB":
        return kb_ai.invoke({"input": user_msg})
    return "I can only answer Math and GK related questions."


In [44]:
dispatcher = RunnableLambda(_dispatcher)

In [45]:
_sessions = {}

In [46]:
def _get_history(session_id: str):
    if session_id not in _sessions:
        _sessions[session_id] = ChatMessageHistory()
    return _sessions[session_id]

In [47]:
cfg_dict = {"configurable":{"session_id":"user1"}}

In [48]:
orchestrator = RunnableWithMessageHistory(
    runnable=dispatcher,
    history_messages_key="history",
    get_session_history=_get_history,
    input_key="input"
)

In [50]:
print(orchestrator.invoke({"input": "what is capital of France"}, config=cfg_dict))



> Entering new AgentExecutor chain...
Could not parse LLM output: `Questions: what is the capital of France
thought ...
Action by using one of the tools access that you have [kb_search]
alwasy finish with final give me an answer to the question

Final answer: The capital of France is Paris.`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE Follow the instructions given to use the toolsCould not parse LLM output: `Questions: what is the capital of France
thought ... 
Action by using one of the tools access that you have [kb_search]
alwasy finish with final give me an answer to the question

Final answer: The capital of France is Paris.`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE Follow the instructions given to use the toolsCould not parse LLM output: `Questions: what is the capital of France
thought ...
Action by using one of the tools access that you have [kb_s